# DuckDB Reporting Benchmarks

In [1]:
import duckdb
import time

# Connect to the DuckDB file
con = duckdb.connect('jaffle_shop.duckdb')
con.sql('SHOW TABLES')

IOException: IO Error: Could not set lock on file "/Users/nicholas_szczerbickyj/Library/CloudStorage/OneDrive-McKinsey&Company/Desktop/PAM/jaffle_shop_duckdb/jaffle_shop.duckdb": Conflicting lock is held in /Users/nicholas_szczerbickyj/.pyenv/versions/3.10.6/bin/python3.10 (PID 23113) by user nicholas_szczerbickyj. See also https://duckdb.org/docs/connect/concurrency

## Preview Tables

In [ ]:
con.sql('SELECT * FROM customers LIMIT 5')

In [ ]:
con.sql('SELECT * FROM orders LIMIT 5')

In [ ]:
con.sql('SELECT * FROM raw_payments LIMIT 5')

## Benchmarking Helper

In [ ]:
def run_and_time_query(query: str):
    start = time.time()
    result = con.execute(query).fetchall()
    end = time.time()
    runtime_ms = round((end - start) * 1000, 2)
    return result, runtime_ms

### 1. Top 10 customers

In [ ]:
query1 = '''SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10'''

In [ ]:
result1, time1 = run_and_time_query(query1)
print(time1)

### 2. Monthly revenue trend

In [ ]:
query2 = '''SELECT DATE_TRUNC('month', order_date) AS month, SUM(orders.amount) AS revenue
FROM orders
JOIN raw_payments USING(order_id)
GROUP BY month
ORDER BY month'''

In [ ]:
result2, time2 = run_and_time_query(query2)
# print(result2)
print(time2)

### 3. Avg order value per customer

In [ ]:
query3 = '''SELECT customer_id, AVG(amount) AS avg_order_value
FROM orders
GROUP BY customer_id'''

In [ ]:
result3, time3 = run_and_time_query(query3)
print(time3)

### 4. Number of orders per customer

In [ ]:
query4 = '''SELECT customer_id, COUNT(order_id) AS num_orders
FROM orders
GROUP BY customer_id
ORDER BY num_orders DESC'''

In [ ]:
result4, time4 = run_and_time_query(query4)
# result4
print(time4)

### 5. Revenue per order

In [ ]:
query5 = '''SELECT order_id, SUM(amount) AS order_revenue
FROM raw_payments
GROUP BY order_id
ORDER BY order_revenue DESC'''

In [ ]:
result5, time5 = run_and_time_query(query5)
# result5
print(time5)

### 6. Active customers per month

In [ ]:
query6 = '''SELECT DATE_TRUNC('month', order_date) AS month, COUNT(DISTINCT customer_id) AS active_customers
FROM orders
GROUP BY month
ORDER BY month'''

In [ ]:
result6, time6 = run_and_time_query(query6)
# result6
print(time6)

### 7. Average payment per order

In [ ]:
query7 = '''SELECT order_id, AVG(amount) AS avg_payment
FROM raw_payments
GROUP BY order_id'''

In [ ]:
result7, time7 = run_and_time_query(query7)
# result7
print(time7)

## Query Runtime Summary

In [ ]:
runtimes = [
    ('Top 10 customers', time1),
    ('Monthly revenue trend', time2),
    ('Avg order value per customer', time3),
    ('Number of orders per customer', time4),
    ('Revenue per order', time5),
    ('Active customers per month', time6),
    ('Average payment per order', time7),
]

print('\nQuery Runtime Summary (ms):')
for name, ms in runtimes:
    print(f'{name:<35} : {ms} ms')

## Polars

In [ ]:
import polars as pl
import os
db_path = os.path.abspath("../jaffle_shop.duckdb")
def run_query_to_polars(query: str):
    start = time.time()
    arrow_table = con.execute(query).arrow()
    df = pl.from_arrow(arrow_table)
    end = time.time()
    runtime_ms = round((end - start) * 1000, 2)
    return df, runtime_ms



In [ ]:
polars_result1, polars_time1 = run_query_to_polars(query1)
polars_result2, polars_time2 = run_query_to_polars(query2)
polars_result3, polars_time3 = run_query_to_polars(query3)
polars_result4, polars_time4 = run_query_to_polars(query4)
polars_result5, polars_time5 = run_query_to_polars(query5)
polars_result6, polars_time6 = run_query_to_polars(query6)
polars_result7, polars_time7 = run_query_to_polars(query7)


In [ ]:
runtimes = [
    ('Top 10 customers by LTV', polars_time1),
    ('Monthly revenue trend', polars_time2),
    ('Avg order value per customer', polars_time3),
    ('Number of orders per customer', polars_time4),
    ('Revenue per order', polars_time5),
    ('Active customers per month', polars_time6),
    ('Average payment per order', polars_time7),
]

print('\nQuery Runtime Summary (ms):')
for name, ms in runtimes:
    print(f'{name:<35} : {ms} ms')

In [ ]:
# polars_result1

# Comparison


In [ ]:
benchmark_comparison = [
    ("Top 10 customers by LTV", time1, polars_time1),
    ("Monthly revenue trend", time2, polars_time2),
    ("Avg order value per customer", time3, polars_time3),
    ("Number of orders per customer", time4, polars_time4),
    ("Revenue per order", time5, polars_time5),
    ("Active customers per month", time6, polars_time6),
    ("Average payment per order", time7, polars_time7)
]
print(f"{'Query':<40} | {'DuckDB (ms)':>12} | {'Polars (ms)':>12} | {'Difference (ms)':>15}")
print("-" * 85)
for name, duck, polars in benchmark_comparison:
    diff = round(duck - polars, 2)
    print(f"{name:<40} | {duck:>12} | {polars:>12} | {diff:>15}")
    ## aligning the columns and choosing the right width
